# Phase 5 — Managed Delta Lakehouse in Databricks (low-cost path)

**Goal:** Run a small, synthetic healthcare Bronze → Silver → Gold Delta pipeline, validate totals and observe tables in Unity Catalog. This requires **no AWS account, no S3 bucket, and no cloud credentials** if using a suitable Databricks Free Edition workspace.

**Cost safety first:** This notebook **starts in preview mode**. The write flag is `False` until you change it intentionally. Use `demo` (10,000 claims), not `medium` (one million) for the first cloud run. Free Edition compute is subject to quotas and feature limits; don't assume a paid workspace is free. This is an educational pipeline, not a production healthcare deployment.

Open the notebook *inside a Databricks Git folder* linked to your public GitHub repository so Python can import `src.cloud_pipeline`.

## Step 1 — Confirm you're in Databricks

Run in Databricks notebooks, not Google Colab. In a Databricks Git folder, the repository root is normally importable. No external pip installs are needed for this managed Delta path.

In [ ]:
import sys
from pathlib import Path
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("Spark:", spark.version)
display(spark.sql("SHOW CATALOGS"))

## Step 2 — Set catalog/schema and safety flag

Choose an existing Unity Catalog catalog from Step 1 (commonly `workspace` in Free Edition). The demonstration schema `healthcare_phase5_demo` must be one you are permitted to create in.

**Do not change the write switch until you're ready**: it will create/overwrite 9 Delta tables in the chosen schema.

In [ ]:
CATALOG = "workspace"  # <-- replace with an existing catalog you can use
SCHEMA = "healthcare_phase5_demo"
SCALE = "demo"
EXECUTE_CLOUD_WRITE = False  # explicit opt-in; starts with NO WRITES

from src.cloud_pipeline import plan, run
spec = plan(CATALOG, SCHEMA, SCALE)
print("Proposed Delta tables:", *spec["tables"], sep="\n - ")
print("Records in demo:", spec["claims"])
print("WRITE ENABLED?", EXECUTE_CLOUD_WRITE)

## Step 3 — Zero-write preview

This checks the settings and prints a run plan. It **does not** create a schema, tables, compute, or S3 objects.

In [ ]:
preview = run(spark, CATALOG, SCHEMA, SCALE, execute=False)
assert preview["status"] == "DRY_RUN"
print(preview)

## Step 4 — Execute the tiny Delta pipeline (optional)

**Only after verifying your Databricks plan and costs**, change `EXECUTE_CLOUD_WRITE` to `True` in Step 2 and rerun Step 2. This cell will otherwise *skip* all writes.

With opt-in, it creates nine managed Delta tables: four Bronze tables, four validated Silver tables, and one Gold claims-by-status table. Tables are overwritten on repeat runs; don't use this schema for any existing valuable data.

In [ ]:
if not EXECUTE_CLOUD_WRITE:
    print("SKIPPED: no cloud writes. Set EXECUTE_CLOUD_WRITE=True in Step 2 to run.")
else:
    result = run(spark, CATALOG, SCHEMA, SCALE, execute=True)
    print(result)
    assert result["status"] == "PASS"
    assert result["claims"] == 10000

## Step 5 — Inspect managed Delta results

The following read-only SQL queries execute **only after** a successful opt-in write. Even read-only queries can consume compute quotas/credits.

In [ ]:
if not EXECUTE_CLOUD_WRITE:
    print("SKIPPED: nothing created.")
else:
    display(spark.sql(f"SELECT claim_status, claim_count, billed_cents FROM `{CATALOG}`.`{SCHEMA}`.gold_claims_by_status"))
    display(spark.sql(f"DESCRIBE DETAIL `{CATALOG}`.`{SCHEMA}`.silver_claims"))
    display(spark.sql(f"DESCRIBE HISTORY `{CATALOG}`.`{SCHEMA}`.silver_claims LIMIT 3"))

## Step 6 — Stop and review spending

Do not configure a repeating schedule by default. Confirm the run status and check Databricks usage/cost view for your plan. The pipeline uses **workspace-managed Delta tables**, not an external S3 bucket. For optional S3 uploads and Unity Catalog external-location setup, see `docs/PHASE5_CLOUD_DEPLOYMENT.md`.

**Result to capture:** table names, claim count, billed amount, Delta format, runtime, and a screenshot showing the successful notebook run. Do not commit secrets to GitHub.